# 02 – Partição, pré-processamento e baseline

Executa depois do notebook 01. Em modo demonstração, a tabela de métricas abaixo vem de dados sintéticos e **não deve ser reportada** como baseline do projeto.

In [1]:
from pathlib import Path
import sys
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ))
import pandas as pd
from src import config
from src.dados import carregar_bruto, limpar, agregar_mensal, validar_totais
from src.demo import gerar_registros_sinteticos, ROTULO_DEMO
pd.set_option("display.width", 140)
from src.variaveis import criar_variaveis
from src.particao import particionar_temporal
from src.preprocessamento import construir_preprocessador, resumo_do_fit
from src.baseline import avaliar_baselines

In [2]:
nome_real = config.PROCESSADO / "base_mensal.csv"
nome_demo = config.PROCESSADO / "base_mensal_DEMO.csv"
MODO_DEMO = not nome_real.exists()
arquivo = nome_demo if MODO_DEMO else nome_real
if MODO_DEMO:
    print(ROTULO_DEMO)
mensal = pd.read_csv(arquivo, parse_dates=[config.COL_MES])
print("Base lida:", arquivo.name, mensal.shape)

Base lida: base_mensal.csv (84, 4)


## Variáveis sem vazamento temporal
Defasagens e média móvel usam só meses anteriores ao mês-alvo.

In [3]:
feats, descartadas = criar_variaveis(mensal)
print("Linhas usadas:", len(feats), "| descartadas por falta de histórico ou alvo ausente:", descartadas)
feats.head()

Linhas usadas: 66 | descartadas por falta de histórico ou alvo ausente: 18


,unidade,mes,exames,municipio,lag_1,lag_2,lag_3,media_movel_3,mes_do_ano,dias_uteis
0,Maurílio de Almeida,2025-11-01,417,NaN,728.0,602.0,262.0,530.666667,11,20
1,Maurílio de Almeida,2025-12-01,324,NaN,417.0,728.0,602.0,582.333333,12,23
2,Maurílio de Almeida,2026-01-01,528,NaN,324.0,417.0,728.0,489.666667,1,22
3,Maurílio de Almeida,2026-02-01,433,NaN,528.0,324.0,417.0,423.000000,2,20
4,Maurílio de Almeida,2026-03-01,573,NaN,433.0,528.0,324.0,428.333333,3,22


## Partição temporal
Treino no passado; validação e teste em meses posteriores. O teste fica guardado.

In [4]:
treino, valid, teste = particionar_temporal(feats)
for nome, d in [("treino", treino), ("validação", valid), ("teste", teste)]:
    print(f"{nome:10s} {d[config.COL_MES].min():%Y-%m} a {d[config.COL_MES].max():%Y-%m}  linhas={len(d)}")

treino     2025-11 a 2026-05  linhas=42
validação  2026-06 a 2026-07  linhas=12
teste      2026-08 a 2026-09  linhas=12


## Pré-processador ajustado só no treino

In [5]:
cols = config.NUMERICAS + config.CATEGORICAS
pre = construir_preprocessador().fit(treino[cols])
resumo = resumo_do_fit(pre)
print("Medianas aprendidas no treino:", resumo["medianas"])
print("Médias aprendidas no treino:", resumo["medias"])
print("Categorias (unidade):", resumo["categorias"]["unidade"][:5], "...")
formas = {n: pre.transform(d[cols]).shape for n, d in [("treino", treino), ("validação", valid), ("teste", teste)]}
print("Formas após transformar:", formas)
assert len({f[1] for f in formas.values()}) == 1, "Conjuntos com número de atributos diferente"

Medianas aprendidas no treino: [165.0, 165.0, 165.0, 172.83333333333334, 22.0]
Médias aprendidas no treino: [260.1190476190476, 255.54761904761904, 242.0, 252.55555555555551, 21.428571428571427]
Categorias (unidade): ['Maurílio de Almeida', 'Particulares', 'Roseane Dore', 'SUS', 'São Lucas'] ...
Formas após transformar: {'treino': (42, 19), 'validação': (12, 19), 'teste': (12, 19)}


## Baselines no recorte de validação
Métricas: MAE, WAPE e viés (previsto − real).

In [6]:
tabela = avaliar_baselines(treino, valid).round(3)
tabela

,baseline,MAE,WAPE,viés
0,Persistência (operacional),63.833,0.258,-18.500
1,Média do treino,50.655,0.205,1.274
2,Mediana do treino,53.083,0.214,-1.083
3,Média móvel de 3 meses,62.333,0.252,10.889
